# train.ipynb — 최종 모델 학습 및 저장

## 이 노트북의 역할 (대회 2차 평가 산출물)

대회 규정(CLAUDE.md 12-3)은 **학습 코드와 추론 코드를 반드시 별도 파일로 분리**할 것을 요구한다.

| 파일 | 하는 일 |
|---|---|
| **`train.ipynb`** (이 파일) | 피처 로딩 → 모델 학습 → **`models/final/`에 저장** |
| `inference.ipynb` | 저장된 모델 로딩 → test 예측 → 제출 파일 생성·검증 |

**두 파일이 어긋나지 않게 하는 방법**은 두 겹이다.

1. **전처리·피처**: CLAUDE.md 8번의 **A방식(parquet 경유)**. `01_preprocessing.ipynb` → `03_features.ipynb`가
   train과 test를 **완전히 같은 함수**로 처리해 `data/processed/*.parquet`에 저장했고,
   두 노트북은 그 parquet만 읽는다. 사이에 전처리 로직이 존재하지 않으므로 어긋날 수 없다.
2. **신경망 구조·손실함수**: `src/nn.py`에 두고 양쪽이 **import**한다.
   노트북 셀에 복사해 두면 한쪽만 고치는 사고가 난다.

## 최종 모델 — 두 모델의 블렌드

| 구성요소 | 무엇 | 근거 |
|---|---|---|
| **모델 1** | LightGBM 분위수 회귀 (τ = 0.70 / 0.50 / 0.65, `actual` 가중) | `phase4_tuning.md` |
| **모델 2** | **MLP + 대회 산식을 직접 최적화하는 손실** (시드 5개 평균) | `phase6_nn_metric_loss.md` |
| **결합** | 그룹별 가중 평균 `w` = 0.8 / 0.5 / 0.9 (MLP 비중) | 교차검증으로 선택 |

**2024 홀드아웃 점수 (정직한 추정치)**

| 설정 | total | 1−NMAE | FICR |
|---|---:|---:|---:|
| LightGBM 단독 | 0.6308 | 0.8636 | 0.3979 |
| MLP 단독 | 0.6435 | 0.8736 | 0.4134 |
| **블렌드 (최종)** | **0.6458** | **0.8744** | **0.4172** |

### 왜 블렌드가 두 축을 모두 개선하는가

Phase 3에서 "단순 평균 앙상블이 FICR을 깎는다"고 기록했다(LightGBM+XGBoost+CatBoost).
지금은 정반대다. **모순이 아니다.**

- 그때: **같은 귀납 편향**(축 정렬 분할)을 가진 트리 셋. 평균이 예측을 평균값 쪽으로 끌어당겼다.
- 지금: **축 정렬 분할(트리) vs 매끄러운 함수(신경망)**. 서로 다른 오차를 내므로 상쇄된다.

Phase 3의 Ridge 실험이 이미 예고했다 — 매끄러운 함수 계열은 NMAE를 따라잡되 FICR에서 뒤진다.
**그 약점을 산식 손실이 메우자, 트리와 상보적인 모델이 됐다.**

## 지킬 규칙

- **랜덤 시드 완전 고정** (CLAUDE.md 12-3). LightGBM은 `deterministic=True` + 스레드 고정,
  PyTorch는 `use_deterministic_algorithms(True)` + **CPU 전용**(GPU는 비결정성 위험).
- **test 데이터는 이 노트북에서 열지도 않는다.**
- 라벨이 없는 행(`kpx_group_3`의 2022년)은 학습에서 제외한다. 0으로 채우지 않는다.
- 표준화 통계(mu, sd)는 **학습 데이터에서만 fit** 하고 저장한다 (CLAUDE.md 4번).

## 0. 준비 — 시드 고정과 상수

In [1]:
import os, random, json, hashlib, platform, subprocess, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
import torch

sys.path.insert(0, ".")
from src.metric import metric, metric_by_group, TARGET_COLS, CAPACITY_KWH
# 신경망 구조·손실함수·표준화는 src/nn.py에서 import (inference.ipynb도 같은 것을 쓴다)
from src.nn import (T_SOFT, MAX_EPOCHS, set_seed, fit_standardizer, standardize,
                    train_mlp, predict_mlp, metric_loss, MLP)

RNG_SEED = 42
set_seed(RNG_SEED)
N_THREADS = 4
torch.set_num_threads(N_THREADS)

PROCESSED_DIR = Path("data/processed")
MODEL_DIR = Path("models/final"); MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ---------- 최종 설정 (근거는 위 마크다운) ----------
SCORE_THRESHOLD = 0.10                       # 채점 대상 기준 (src/metric.py와 동일)
GROUP_TAU = {"kpx_group_1": 0.70, "kpx_group_2": 0.50, "kpx_group_3": 0.65}   # LightGBM 분위수
BLEND_W = {"kpx_group_1": 0.8, "kpx_group_2": 0.5, "kpx_group_3": 0.9}        # MLP 비중
NN_SEEDS = [42, 1337, 2024, 7, 99]           # 신경망은 분산이 커서 시드 평균이 효과적이다

# 트리 개수·에폭 수를 정하기 위한 내부검증 구간 (학습에 쓰지 않은 최근 6개월)
INNER_VAL_START = pd.Timestamp("2024-07-01 01:00:00")
MAX_ROUNDS, EARLY_STOP = 2000, 50

LGB_PARAMS = dict(objective="quantile", learning_rate=0.05, num_leaves=63, min_data_in_leaf=40,
                  feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
                  verbosity=-1, seed=RNG_SEED, num_threads=N_THREADS,
                  deterministic=True, force_row_wise=True)

print(f"Python {platform.python_version()} | LightGBM {lgb.__version__} | PyTorch {torch.__version__}")
print(f"OS {platform.platform()}")
print(f"시드 {RNG_SEED} | 스레드 {N_THREADS} | T_SOFT {T_SOFT} | 블렌드 w {BLEND_W}")

Python 3.13.14 | LightGBM 4.6.0 | PyTorch 2.13.0+cpu
OS Windows-11-10.0.26200-SP0
시드 42 | 스레드 4 | T_SOFT 0.006 | 블렌드 w {'kpx_group_1': 0.8, 'kpx_group_2': 0.5, 'kpx_group_3': 0.9}


## 1. 피처 로딩

`03_features.ipynb`가 저장한 parquet만 읽는다. 원본 CSV를 다시 읽지 않는다.

In [2]:
feat = pd.read_parquet(PROCESSED_DIR / "features_train.parquet")
dtm = feat["forecast_kst_dtm"]
FEATURE_COLS = [c for c in feat.columns if c not in TARGET_COLS + ["forecast_kst_dtm"]]

print(f"features_train.parquet: {feat.shape} | 피처 {len(FEATURE_COLS)}개")
print(f"기간: {dtm.min()} ~ {dtm.max()}\n")
for g in TARGET_COLS:
    n_all = int(feat[g].notna().sum())
    n_sc = int((feat[g] >= CAPACITY_KWH[g] * SCORE_THRESHOLD).sum())
    print(f"  {g}: 라벨 {n_all:,}개 | 채점 대상 {n_sc:,}개 ({n_sc/n_all*100:.1f}%)")

assert feat[FEATURE_COLS].isna().sum().sum() == 0
assert not np.isinf(feat[FEATURE_COLS].to_numpy()).any()

features_train.parquet: (26304, 183) | 피처 179개
기간: 2022-01-01 01:00:00 ~ 2025-01-01 00:00:00

  kpx_group_1: 라벨 26,200개 | 채점 대상 15,915개 (60.7%)
  kpx_group_2: 라벨 26,201개 | 채점 대상 15,891개 (60.7%)
  kpx_group_3: 라벨 17,538개 | 채점 대상 9,414개 (53.7%)


## 2. 학습 기간과 마스크

검증(2024 홀드아웃)이 끝났으므로, 최종 모델은 **2022-01-01 ~ 2024-12-31의 모든 라벨**로 학습한다.

**트리 개수·에폭 수를 정하는 방법** (두 모델 공통):

1. 2024-07-01 이전으로 학습하고 **2024년 하반기(내부검증)** 에서 조기 종료
2. LightGBM은 데이터가 늘어난 비율만큼 트리 수를 키우고, MLP는 그 에폭 수를 그대로 쓴다
3. 그 개수로 **전체 기간**을 다시 학습

**두 모델의 학습 데이터가 다르다**는 점에 주의한다.

| 모델 | 학습 행 | 왜 |
|---|---|---|
| LightGBM | 라벨 있는 **모든** 행 + `actual` 표본 가중 | `actual` 가중이 저발전 시간을 자동으로 지운다 (`phase4_tuning.md` §3-4-1) |
| MLP | **채점 대상 행만** (이용률 ≥ 10%) | 산식 손실의 정의역이 채점 대상 시간이기 때문 |

In [3]:
def build_masks(group):
    """한 그룹의 학습/내부검증 마스크. scored=채점 대상(실제 이용률 >= 10%)."""
    lab = feat[group].notna()
    sc = feat[group] >= CAPACITY_KWH[group] * SCORE_THRESHOLD
    return {
        "fit_all": lab,
        "fit_scored": lab & sc,
        "inner_tr": lab & (dtm < INNER_VAL_START),
        "inner_tr_scored": lab & sc & (dtm < INNER_VAL_START),
        "inner_va": lab & (dtm >= INNER_VAL_START),
        "inner_va_scored": lab & sc & (dtm >= INNER_VAL_START),
    }


for g in TARGET_COLS:
    m = build_masks(g)
    print(f"{g}: LGB 최종학습 {int(m['fit_all'].sum()):,} | MLP 최종학습(채점행) {int(m['fit_scored'].sum()):,} "
          f"| 내부검증 {int(m['inner_va'].sum()):,} (채점행 {int(m['inner_va_scored'].sum()):,})")

kpx_group_1: LGB 최종학습 26,200 | MLP 최종학습(채점행) 15,915 | 내부검증 4,410 (채점행 2,443)
kpx_group_2: LGB 최종학습 26,201 | MLP 최종학습(채점행) 15,891 | 내부검증 4,410 (채점행 2,397)
kpx_group_3: LGB 최종학습 17,538 | MLP 최종학습(채점행) 9,414 | 내부검증 4,410 (채점행 2,159)


## 3. 모델 1 — LightGBM 분위수 회귀

`phase4_tuning.md`에서 확정한 설정 그대로다.

- **`actual` 표본 가중**: FICR이 `Σ actual·단가 / Σ actual·4`이므로 산식과 일치한다.
- **분위수 τ > 0.5**: 채점 대상 여부가 `actual`로 정해지므로 추정 대상이 `E[y|x]`가 아니라
  `E[y | x, y ≥ 0.1×용량]`이다. 그 값은 항상 더 크다.

In [4]:
def train_lgb(group):
    """내부검증으로 트리 수 결정 -> 데이터 증가 비율만큼 확대 -> 전체 재학습."""
    params = {**LGB_PARAMS, "alpha": GROUP_TAU[group]}
    m = build_masks(group)

    d_tr = lgb.Dataset(feat.loc[m["inner_tr"], FEATURE_COLS], label=feat.loc[m["inner_tr"], group],
                       weight=feat.loc[m["inner_tr"], group].to_numpy())
    d_va = lgb.Dataset(feat.loc[m["inner_va_scored"], FEATURE_COLS],
                       label=feat.loc[m["inner_va_scored"], group], reference=d_tr)
    probe = lgb.train(params, d_tr, MAX_ROUNDS, valid_sets=[d_va],
                      callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False)])

    ratio = m["fit_all"].sum() / m["inner_tr"].sum()
    n_final = max(int(round(probe.best_iteration * ratio)), 50)

    d_full = lgb.Dataset(feat.loc[m["fit_all"], FEATURE_COLS], label=feat.loc[m["fit_all"], group],
                         weight=feat.loc[m["fit_all"], group].to_numpy())
    booster = lgb.train(params, d_full, n_final)
    return booster, {"best_iter_inner": int(probe.best_iteration), "n_trees": int(booster.num_trees()),
                     "alpha": GROUP_TAU[group]}


t0 = time.time()
boosters, lgb_info = {}, {}
for g in TARGET_COLS:
    boosters[g], lgb_info[g] = train_lgb(g)
    i = lgb_info[g]
    print(f"{g}: τ={i['alpha']:.2f} | 내부검증 {i['best_iter_inner']}그루 -> 최종 {i['n_trees']}그루")
print(f"\nLightGBM 학습 {time.time()-t0:.1f}초")

kpx_group_1: τ=0.70 | 내부검증 93그루 -> 최종 112그루


kpx_group_2: τ=0.50 | 내부검증 165그루 -> 최종 198그루


kpx_group_3: τ=0.65 | 내부검증 82그루 -> 최종 110그루

LightGBM 학습 9.1초


## 4. 모델 2 — MLP + 대회 산식 직접 최적화

`src/nn.py`의 `metric_loss()`가 손실이다. 계단 함수인 FICR 단가를 시그모이드로 근사해
`0.5·(1−NMAE) + 0.5·FICR`을 **그대로 최대화**한다.

**시드 5개를 평균한다**: 신경망은 초기값에 따라 결과가 흔들린다(트리와 달리).
같은 설정으로 5번 학습해 예측을 평균하면 분산이 줄어든다.
(LightGBM에서는 시드 앙상블이 도움이 되지 않았다 — 이미 배깅으로 분산이 최소였기 때문이다.)

**에폭 수 결정**: 내부검증(2024 하반기)의 **대회 산식**으로 조기 종료한다.
시드마다 최적 에폭이 다르므로 그 **중앙값**을 최종 에폭으로 쓴다.

In [5]:
from src.metric import group_score


def train_mlp_group(group):
    """
    내부검증으로 에폭 수를 정한 뒤, 전체 기간으로 시드 5개를 학습해 예측을 평균한다.
    출력: (모델 리스트, mu, sd, 정보 dict)
    """
    cap = CAPACITY_KWH[group]
    m = build_masks(group)

    # --- 1단계: 내부검증으로 에폭 수 결정 ---
    X_itr = feat.loc[m["inner_tr_scored"], FEATURE_COLS].to_numpy(np.float32)
    y_itr = (feat.loc[m["inner_tr_scored"], group] / cap).to_numpy(np.float32)
    X_iva = feat.loc[m["inner_va_scored"], FEATURE_COLS].to_numpy(np.float32)
    y_iva = (feat.loc[m["inner_va_scored"], group] / cap).to_numpy(np.float32)

    mu_i, sd_i = fit_standardizer(X_itr)                 # 내부학습에서만 fit
    Xi, Xv = (X_itr - mu_i) / sd_i, standardize(X_iva, mu_i, sd_i)

    def eval_fn(model):
        """조기 종료 기준 = 내부검증의 대회 산식 (MAE가 아니다)."""
        with torch.no_grad():
            p = np.clip(model(Xv).numpy(), 0, 1)
        return group_score(y_iva * cap, p * cap, cap)[0]

    epochs = []
    for s in NN_SEEDS:
        _, be = train_mlp(Xi, y_itr, seed=s, n_epochs=MAX_EPOCHS, eval_fn=eval_fn)
        epochs.append(be)
    ep_final = int(np.median(epochs))

    # --- 2단계: 전체 기간으로 재학습 (조기 종료 없이 ep_final 에폭) ---
    X_full = feat.loc[m["fit_scored"], FEATURE_COLS].to_numpy(np.float32)
    y_full = (feat.loc[m["fit_scored"], group] / cap).to_numpy(np.float32)
    mu, sd = fit_standardizer(X_full)                    # 최종 학습 데이터에서만 fit
    Xf = (X_full - mu) / sd

    models = [train_mlp(Xf, y_full, seed=s, n_epochs=ep_final, eval_fn=None)[0] for s in NN_SEEDS]
    info = {"epochs_inner": epochs, "n_epochs": ep_final, "n_seeds": len(NN_SEEDS),
            "n_train_rows": int(m["fit_scored"].sum())}
    return models, mu, sd, info


t0 = time.time()
mlp_models, scalers, mlp_info = {}, {}, {}
for g in TARGET_COLS:
    mlp_models[g], mu, sd, mlp_info[g] = train_mlp_group(g)
    scalers[g] = (mu, sd)
    i = mlp_info[g]
    print(f"{g}: 내부검증 에폭 {i['epochs_inner']} -> 최종 {i['n_epochs']}에폭 × 시드 {i['n_seeds']}개 "
          f"| 학습 {i['n_train_rows']:,}행(채점 대상)")
print(f"\nMLP 학습 {time.time()-t0:.1f}초")

kpx_group_1: 내부검증 에폭 [91, 36, 71, 36, 36] -> 최종 36에폭 × 시드 5개 | 학습 15,915행(채점 대상)


kpx_group_2: 내부검증 에폭 [91, 46, 46, 141, 51] -> 최종 51에폭 × 시드 5개 | 학습 15,891행(채점 대상)


kpx_group_3: 내부검증 에폭 [71, 21, 21, 16, 51] -> 최종 21에폭 × 시드 5개 | 학습 9,414행(채점 대상)

MLP 학습 268.4초


## 5. 블렌드와 학습 데이터 위 점검

**학습에 쓴 데이터로 낸 점수는 성능이 아니다.** 과적합돼 있다.
여기서 보는 것은 오직 하나 — 모델이 정상적으로 학습됐는가(예측값이 상식적인 범위인가).

진짜 성능 추정치는 **2024 홀드아웃 0.6458**이며, 그때는 2024를 학습에서 뺀 상태였다
(`reports/phase6_nn_metric_loss.md`).

In [6]:
X_all = feat[FEATURE_COLS].to_numpy(np.float32)

pred_lgb, pred_mlp, pred_blend = {}, {}, {}
for g in TARGET_COLS:
    cap = CAPACITY_KWH[g]
    pred_lgb[g] = np.clip(boosters[g].predict(feat[FEATURE_COLS]), 0, cap)
    mu, sd = scalers[g]
    pred_mlp[g] = np.mean([predict_mlp(m_, X_all, mu, sd, cap) for m_ in mlp_models[g]], axis=0)
    w = BLEND_W[g]
    pred_blend[g] = np.clip((1 - w) * pred_lgb[g] + w * pred_mlp[g], 0, cap)

for name, pr in [("LightGBM", pred_lgb), ("MLP", pred_mlp), ("블렌드", pred_blend)]:
    t, n, f = metric(feat[TARGET_COLS], pd.DataFrame(pr))
    print(f"[학습 데이터 위, 성능 지표 아님] {name:<9}: total {t:.4f} | 1-NMAE {n:.4f} | FICR {f:.4f}")

print("\n[예측 이용률 분포 점검]")
for g in TARGET_COLS:
    cap = CAPACITY_KWH[g]
    cf = pred_blend[g] / cap
    act = (feat[g] / cap).dropna()
    print(f"  {g}: 예측 평균 {cf.mean():.3f} (실제 {act.mean():.3f}) | 최소 {cf.min():.3f} 최대 {cf.max():.3f}")
print("\n예측 평균이 실제보다 높은 것은 정상이다 — 채점 대상이 actual 기준이라 위로 편향해야 한다")
print("(phase4_tuning.md §1-2). MLP는 채점 대상 행만 학습하므로 더욱 그렇다.")

[학습 데이터 위, 성능 지표 아님] LightGBM : total 0.7515 | 1-NMAE 0.8864 | FICR 0.6166
[학습 데이터 위, 성능 지표 아님] MLP      : total 0.6586 | 1-NMAE 0.8730 | FICR 0.4442
[학습 데이터 위, 성능 지표 아님] 블렌드      : total 0.6844 | 1-NMAE 0.8799 | FICR 0.4889

[예측 이용률 분포 점검]
  kpx_group_1: 예측 평균 0.400 (실제 0.307) | 최소 0.072 최대 0.974
  kpx_group_2: 예측 평균 0.401 (실제 0.328) | 최소 0.028 최대 0.976
  kpx_group_3: 예측 평균 0.363 (실제 0.265) | 최소 0.071 최대 0.957

예측 평균이 실제보다 높은 것은 정상이다 — 채점 대상이 actual 기준이라 위로 편향해야 한다
(phase4_tuning.md §1-2). MLP는 채점 대상 행만 학습하므로 더욱 그렇다.


## 6. 모델 저장

| 파일 | 내용 |
|---|---|
| `lgbm_{group}.txt` | LightGBM 부스터 |
| `mlp_{group}_seed{s}.pt` | MLP 가중치 (그룹당 5개) |
| `scaler_{group}.npz` | 표준화 통계 (mu, sd) — **학습 데이터에서만 fit한 값** |
| `config.json` | 피처 목록·순서, τ, 블렌드 w, 시드, 에폭, 해시, 버전 |

**피처 목록을 순서까지 저장하는 이유**: LightGBM도 MLP도 컬럼 **이름이 아니라 위치**로 읽는다.
`inference.ipynb`가 다른 순서의 표를 넣으면 **오류 없이 조용히 엉뚱한 예측**을 낸다.
그래서 추론 때 대조해 다르면 즉시 멈추게 한다.

In [7]:
def sha256_of(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()[:16]


def git_state():
    """읽기 전용 git 명령만 쓴다 (CLAUDE.md 3번)."""
    try:
        h = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
        d = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True).stdout.strip()
        return f"{h}({'dirty' if d else 'clean'})"
    except Exception:
        return "unknown"


hashes = {}
for g in TARGET_COLS:
    p = MODEL_DIR / f"lgbm_{g}.txt"
    boosters[g].save_model(str(p))
    hashes[f"lgbm_{g}"] = sha256_of(p)

    mu, sd = scalers[g]
    np.savez(MODEL_DIR / f"scaler_{g}.npz", mu=mu, sd=sd)
    hashes[f"scaler_{g}"] = sha256_of(MODEL_DIR / f"scaler_{g}.npz")

    for s, model in zip(NN_SEEDS, mlp_models[g]):
        pth = MODEL_DIR / f"mlp_{g}_seed{s}.pt"
        torch.save(model.state_dict(), pth)
        hashes[f"mlp_{g}_seed{s}"] = sha256_of(pth)

config = {
    "created": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
    "git_hash": git_state(),
    "model": "blend(lightgbm_quantile, mlp_metric_loss)",
    "feature_source": "data/processed/features_train.parquet (v1, 03_features.ipynb)",
    "feature_cols": FEATURE_COLS, "n_features": len(FEATURE_COLS),
    "group_tau": GROUP_TAU, "blend_w": BLEND_W,
    "nn_seeds": NN_SEEDS, "t_soft": T_SOFT,
    "score_threshold": SCORE_THRESHOLD,
    "lgb_params": {k: v for k, v in LGB_PARAMS.items() if k != "alpha"},
    "lgb_info": lgb_info, "mlp_info": mlp_info,
    "train_period": [str(dtm.min()), str(dtm.max())],
    "sha256": hashes, "seed": RNG_SEED, "n_threads": N_THREADS,
    "versions": {"python": platform.python_version(), "lightgbm": lgb.__version__,
                 "torch": torch.__version__, "numpy": np.__version__,
                 "pandas": pd.__version__, "os": platform.platform()},
    "holdout_score_2024": {"total": 0.6458, "one_minus_nmae": 0.8744, "ficr": 0.4172,
                           "note": "phase6_nn_metric_loss.md, 2024를 학습에서 뺀 상태의 정직한 점수"},
}
with open(MODEL_DIR / "config.json", "w", encoding="utf-8") as fp:
    json.dump(config, fp, ensure_ascii=False, indent=2)

n_files = len(list(MODEL_DIR.glob("*")))
total_mb = sum(p.stat().st_size for p in MODEL_DIR.glob("*")) / 1e6
print(f"저장 완료: {MODEL_DIR} ({n_files}개 파일, {total_mb:.1f} MB)")
print(f"git: {config['git_hash']}")
for k in list(hashes)[:3]:
    print(f"  {k}: sha256[:16]={hashes[k]}")

저장 완료: models\final (22개 파일, 9.9 MB)
git: 4248c9a(dirty)
  lgbm_kpx_group_1: sha256[:16]=225af566dcff2cf1
  scaler_kpx_group_1: sha256[:16]=d7f885be4a79e0e8
  mlp_kpx_group_1_seed42: sha256[:16]=48ac92a4befa6710


## 7. 재현성 검증 — 처음부터 다시 학습해 같은 예측이 나오는가

CLAUDE.md 12-3의 핵심 요건: **"학습 재실행 시 동일한 제출 파일이 나오는지 최종 단계에서 실제로 검증한다."**

LightGBM은 `deterministic=True` + 스레드 고정으로,
PyTorch는 `use_deterministic_algorithms(True)` + CPU + 시드 고정으로 비트 단위 재현을 노린다.

In [8]:
print("재학습 중... (LightGBM + MLP 전부)")
ok = True
for g in TARGET_COLS:
    cap = CAPACITY_KWH[g]

    b2, _ = train_lgb(g)
    p1 = boosters[g].predict(feat[FEATURE_COLS]); p2 = b2.predict(feat[FEATURE_COLS])
    same_lgb = np.array_equal(p1, p2)

    m2, mu2, sd2, _ = train_mlp_group(g)
    q1 = np.mean([predict_mlp(m_, X_all, *scalers[g], cap) for m_ in mlp_models[g]], axis=0)
    q2 = np.mean([predict_mlp(m_, X_all, mu2, sd2, cap) for m_ in m2], axis=0)
    same_mlp = np.allclose(q1, q2, atol=0, rtol=0)

    ok &= same_lgb and same_mlp
    print(f"  {g}: LGB 동일={same_lgb} (최대차 {np.abs(p1-p2).max():.3e}) | "
          f"MLP 동일={same_mlp} (최대차 {np.abs(q1-q2).max():.3e})")

print(f"\n재현성: {'✔ 두 번 실행 결과가 완전히 같다' if ok else '✘ 다르다 — 비결정 요소를 제거해야 한다'}")
assert ok, "재현성 실패: 시드/스레드/deterministic 설정을 확인하세요"

print(f"\n{'='*76}")
print("학습 완료. 다음 단계: inference.ipynb 를 실행해 제출 파일을 만든다.")
print(f"{'='*76}")

재학습 중... (LightGBM + MLP 전부)


  kpx_group_1: LGB 동일=True (최대차 0.000e+00) | MLP 동일=True (최대차 0.000e+00)


  kpx_group_2: LGB 동일=True (최대차 0.000e+00) | MLP 동일=True (최대차 0.000e+00)


  kpx_group_3: LGB 동일=True (최대차 0.000e+00) | MLP 동일=True (최대차 0.000e+00)

재현성: ✔ 두 번 실행 결과가 완전히 같다

학습 완료. 다음 단계: inference.ipynb 를 실행해 제출 파일을 만든다.
